# Data Cleaning and Validation - Manjil

In [17]:
print(df_final.groupby('month')['precipitation_mm'].mean())
print(df_final['pm25'].notna().sum(), "rows have pm25")
print(df_final['pm1'].notna().sum(), "rows have pm1")

month
Apr     0.909234
Aug     9.622444
Dec     0.039013
Feb     0.370706
Jan     0.040085
Jul    11.697000
Jun     8.773589
Mar     2.994728
May     4.177445
Nov     0.128091
Oct     3.925206
Sep     6.567704
Name: precipitation_mm, dtype: float64
6659 rows have pm25
3990 rows have pm1


In [18]:
# Does low rainfall correlate with high PM2.5? (your core climate→environment claim)
monthly_avg = df_final.groupby('month').agg(
    avg_pm25=('pm25','mean'),
    avg_precip=('precipitation_mm','mean')
).reindex(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
print(monthly_avg)

        avg_pm25  avg_precip
month                       
Jan    87.656442    0.040085
Feb    81.171489    0.370706
Mar    81.184279    2.994728
Apr    90.799115    0.909234
May    41.338498    4.177445
Jun    28.399505    8.773589
Jul    16.882857   11.697000
Aug    15.782603    9.622444
Sep    20.264563    6.567704
Oct    37.546470    3.925206
Nov    64.567945    0.128091
Dec    93.858333    0.039013


In [19]:
#Check for impossible/negative values (sensor errors)
print(df_final[['temperature_c','precipitation_mm','humidity_pct','pm25','pm1','o3']].describe())

       temperature_c  precipitation_mm  humidity_pct         pm25  \
count    7155.000000       7155.000000   7155.000000  6659.000000   
mean       17.693898          2.695680     71.571948    62.399943   
std         5.046883          9.034856     16.193257    48.278787   
min         1.630000          0.000000     16.320000     0.000000   
25%        13.600000          0.000000     63.560000    31.100000   
50%        17.440000          0.000000     74.540000    53.500000   
75%        22.210000          0.725000     83.960000    80.700000   
max        32.050000        233.660000     97.140000   438.000000   

               pm1           o3  
count  3990.000000  2262.000000  
mean     41.067154    -0.439045  
std      26.985869     0.468027  
min       0.000000    -0.999000  
25%      23.900000    -0.999000  
50%      35.700000    -0.216000  
75%      50.900000     0.017800  
max     197.000000     0.075900  


In [20]:
print((df_final['pm25'] == 0).sum())

1


In [22]:
print(df_final.isna().sum())

location_name          0
date                   0
temperature_c          0
precipitation_mm       0
humidity_pct           0
o3                  4893
pm1                 3165
pm10                6676
pm25                 496
relativehumidity    3165
temperature         2998
um003               3165
year                   0
month                  0
tourist_arrivals       0
dtype: int64


In [23]:
print(df_final.isna().mean().round(3) * 100)  # percentage missing per column

location_name        0.0
date                 0.0
temperature_c        0.0
precipitation_mm     0.0
humidity_pct         0.0
o3                  68.4
pm1                 44.2
pm10                93.3
pm25                 6.9
relativehumidity    44.2
temperature         41.9
um003               44.2
year                 0.0
month                0.0
tourist_arrivals     0.0
dtype: float64


In [24]:
df_final_clean = df_final[['location_name','date','year','month',
                             'temperature_c','precipitation_mm','humidity_pct',
                             'pm25','pm1','tourist_arrivals']]
df_final_clean = df_final_clean.rename(columns={
    'temperature_c':'temperature',
    'precipitation_mm':'rainfall',
    'humidity_pct':'humidity'
})
print(df_final_clean.shape)
print(df_final_clean.isna().sum())

(7155, 10)
location_name          0
date                   0
year                   0
month                  0
temperature            0
rainfall               0
humidity               0
pm25                 496
pm1                 3165
tourist_arrivals       0
dtype: int64


In [25]:
print("Exact duplicates:", df_final_clean.duplicated().sum())
print("Duplicate station+date:", df_final_clean.duplicated(subset=['location_name','date']).sum())

for col in ['temperature','rainfall','humidity','pm25','pm1']:
    neg = (df_final_clean[col] < 0).sum()
    print(f"{col}: {neg} negative values")

print("Humidity below 0:", (df_final_clean['humidity'] < 0).sum())
print("Humidity above 100:", (df_final_clean['humidity'] > 100).sum())

Exact duplicates: 0
Duplicate station+date: 0
temperature: 0 negative values
rainfall: 0 negative values
humidity: 0 negative values
pm25: 0 negative values
pm1: 0 negative values
Humidity below 0: 0
Humidity above 100: 0


In [26]:
df_final_clean

,location_name,date,year,month,temperature,rainfall,humidity,pm25,pm1,tourist_arrivals
0,Embassy Kathmandu,2021-01-01,2021,Jan,13.61,0.00,43.79,109.0,NaN,8874
1,Embassy Kathmandu,2021-01-02,2021,Jan,13.90,0.00,42.38,131.0,NaN,8874
2,Embassy Kathmandu,2021-01-03,2021,Jan,15.05,0.00,42.59,105.0,NaN,8874
3,Embassy Kathmandu,2021-01-04,2021,Jan,16.34,0.00,47.81,265.0,NaN,8874
4,Embassy Kathmandu,2021-01-05,2021,Jan,16.47,0.00,53.05,258.0,NaN,8874
...,...,...,...,...,...,...,...,...,...,...
7150,CEN-SR-19 : Bharatpur Ward 02,2025-12-27,2025,Dec,14.09,0.00,71.92,134.0,72.5,98180
7151,CEN-SR-19 : Bharatpur Ward 02,2025-12-28,2025,Dec,12.95,0.00,74.49,80.3,43.8,98180
7152,CEN-SR-19 : Bharatpur Ward 02,2025-12-29,2025,Dec,12.34,0.21,71.78,72.2,41.6,98180
7153,CEN-SR-19 : Bharatpur Ward 02,2025-12-30,2025,Dec,11.50,0.02,66.73,63.0,37.5,98180


In [28]:
df_final_clean.to_csv('grdv_nepal_t1_dataset_cleaned.csv', index=False)

In [29]:
print("Saved:", df_final_clean.shape)

Saved: (7155, 10)
